# Kazakh profanity detection — Colab runner

Runs the whole project in a fresh Colab runtime: install → (optional) data preparation → training → one test evaluation → predictions on your own text.

1. Put your repository link in `REPO_URL` below.
2. `Runtime → Run all`. No GPU is needed (training takes a few seconds on CPU).

All scripts run with `!python -m ...` in a subprocess, so the pinned package versions are used without restarting the runtime.

In [ ]:
REPO_URL = "https://github.com/<your-username>/kazakh-profanity-detection"  # <- change me

import os
if not os.path.exists("kazakh-profanity-detection"):
    !git clone -q {REPO_URL} kazakh-profanity-detection
%cd kazakh-profanity-detection
!python --version

In [ ]:
!pip install -q -r requirements.txt

## 1. Data preparation (optional)

The cleaned train/val/test splits are already in `data/processed/`, so you can skip this step.
To rebuild them from the course file, set `REBUILD_DATA = True` and upload `kazakh_profanity_share.xlsx` when asked.

In [ ]:
REBUILD_DATA = False

if REBUILD_DATA:
    from google.colab import files
    uploaded = files.upload()                       # choose kazakh_profanity_share.xlsx
    name = next(iter(uploaded))
    os.makedirs("data/raw", exist_ok=True)
    os.replace(name, "data/raw/kazakh_profanity_share.xlsx")
    !python -m src.prepare_data
else:
    !cat results/data_stats.json

## 2. Train (settings and threshold are chosen on the validation set only)

In [ ]:
!python -m src.train
!cat models/model_meta.json

## 3. Evaluate once on the held-out test set

In [ ]:
!python -m src.evaluate
from IPython.display import Image, display
display(Image("results/confusion_matrix.png"))

## 4. Try your own comment

Edit `text` and run the cell. `--explain` shows the normalized text and the character n-grams that pushed the score up or down.

In [ ]:
text = "Сапасы өте нашар, ақшама өкінемін"   # <- type any Kazakh comment here
!python -m src.predict "{text}" --explain

In [ ]:
# several comments at once
!python -m src.predict "Жеткізу тез болды, рахмет!" "Kotakbas dukan" "Мал сататын жер екен"